## Mean test pass rate

In [10]:
import pandas as pd

df = pd.read_csv("../results/raw_csv/master_analysis.csv")

table = (
    df[df["pass_rate"].notna()]
      .groupby(["model","generation"])["pass_rate"]
      .mean()
      .reset_index()
)

table.to_csv("../results/tables/mean_pass_rate_model_gen.csv", index=False)
table

,model,generation,pass_rate
0,gemma,gen_1,0.464980
1,gemma,gen_2,0.458803
2,gemma,gen_3,0.478618
3,human,soln1,0.831757
4,human,soln2,0.827629
5,llama,gen_1,0.423199
6,llama,gen_2,0.426447
7,llama,gen_3,0.425754
8,qwen,gen_1,0.360750
9,qwen,gen_2,0.382279


### Pass rate vs vulnerability count

In [11]:
master = pd.read_csv("../results/raw_csv/master_analysis.csv")
master.head(2)

,sample_id,model,generation,batch,problem_key,problem_key_source,shard,compile_attempted,compiled,has_tests,...,cppcheck_has_cwe,cppcheck_errors,cppcheck_style,cppcheck_perf,clang_total,clang_warnings,clang_errors,clang_bugprone,clang_performance,clang_analyzer
0,human/soln1/00000/0029_331_E2._Deja_Vu.cpp,human,soln1,00000,331_E2_Deja_Vu,json,0,1,1,1,...,1,0,4,0,30,30,0,15,1,1
1,human/soln1/00000/0032_39_H._Multiplication_Ta...,human,soln1,00000,39_H_Multiplication_Table,json,0,1,1,1,...,0,0,0,0,3,3,0,0,1,0


In [12]:
cppcheck = pd.read_csv("../results/raw_csv/cppcheck_detailed.csv")
cppcheck.head(2)

,sample_id,model,generation,batch,problem_key,error_id,severity,cwe,message
0,human/soln1/00000/0146_p03194_CADDi_2018_for_B...,human,soln1,00000,p03194 caddi 2018 for beginners product and gcd,variableScope,style,398.0,The scope of the variable 'k' can be reduced.
1,human/soln1/00000/0266_1285_C._Fadi_and_LCM,human,soln1,00000,1285 c fadi and lcm,variableScope,style,398.0,The scope of the variable 'a' can be reduced.


In [13]:
# master_analysis already has cppcheck columns merged
# cppcheck_total, cppcheck_warnings, cppcheck_cwe_count, cppcheck_has_cwe

merged = master[["sample_id", "model", "generation", "pass_rate", 
                 "cppcheck_total", "cppcheck_cwe_count"]].copy()
merged = merged[merged["pass_rate"].notna()]
merged.head(2)

,sample_id,model,generation,pass_rate,cppcheck_total,cppcheck_cwe_count
0,human/soln1/00000/0029_331_E2._Deja_Vu.cpp,human,soln1,1.000000,4,1
1,human/soln1/00000/0032_39_H._Multiplication_Ta...,human,soln1,0.142857,0,0


# TPR with TAGS

In [14]:
import pandas as pd

master = pd.read_csv("../results/raw_csv/master_analysis.csv")
master

,sample_id,model,generation,batch,problem_key,problem_key_source,shard,compile_attempted,compiled,has_tests,...,cppcheck_has_cwe,cppcheck_errors,cppcheck_style,cppcheck_perf,clang_total,clang_warnings,clang_errors,clang_bugprone,clang_performance,clang_analyzer
0,human/soln1/00000/0029_331_E2._Deja_Vu.cpp,human,soln1,00000,331_E2_Deja_Vu,json,0,1,1,1,...,1,0,4,0,30,30,0,15,1,1
1,human/soln1/00000/0032_39_H._Multiplication_Ta...,human,soln1,00000,39_H_Multiplication_Table,json,0,1,1,1,...,0,0,0,0,3,3,0,0,1,0
2,human/soln1/00000/0223_937_D._Sleepy_Game.cpp,human,soln1,00000,937_D_Sleepy_Game,json,0,1,1,1,...,0,0,0,0,6,6,0,1,0,1
3,human/soln1/00000/0177_1208_A._XORinacci.cpp,human,soln1,00000,1208_A_XORinacci,json,0,1,1,1,...,0,0,0,0,5,5,0,2,1,0
4,human/soln1/00000/0122_551_B._ZgukistringZ.cpp,human,soln1,00000,551_B_ZgukistringZ,json,0,1,1,1,...,1,0,4,1,3,3,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8913,llama/gen_2/00002_2/0052_136_B._Ternary_Logic.cpp,llama,gen_2,00002_2,136_B_Ternary_Logic,json,2,1,1,1,...,0,0,0,0,2,2,0,0,1,0
8914,llama/gen_2/00002_2/0220_1198_A._MP3.cpp,llama,gen_2,00002_2,1198_A_MP3,json,2,1,1,1,...,1,0,4,0,6,6,0,1,2,1
8915,llama/gen_2/00002_2/0248_436_B._Om_Nom_and_Spi...,llama,gen_2,00002_2,436_B_Om_Nom_and_Spiders,json,2,1,1,1,...,0,0,0,0,3,3,0,1,1,0
8916,llama/gen_2/00002_2/0127_110_B._Lucky_String.cpp,llama,gen_2,00002_2,110_B_Lucky_String,json,2,1,1,1,...,0,0,0,0,2,2,0,0,1,0


In [ ]:
import pandas as pd

master = pd.read_csv("../results/raw_csv/master_analysis.csv")
tags = pd.read_csv("../results/raw_csv/problem_tags.csv")
master
# merge on problem_key
#df = master.merge(tags[["problem_key", "tags"]], on="problem_key", how="left")

: 

## TPR for Human - Total ~ pivot

In [ ]:
agg = (
    df[df.model == "human"]
      .groupby(["tag", "generation"])
      .agg(
          mean_pass_rate=("pass_rate", "mean"),
          n=("pass_rate", "count")
      )
      .reset_index()
)

# keep only sufficiently large groups
agg = agg[agg.n >= 15].sort_values(["tag", "generation"])

agg.to_csv("../results/tables/human_mean_pass_rate_by_tag_and_gen.csv",
           index=False)

pivot = (
    agg.pivot(index="tag", columns="generation", values="mean_pass_rate")
       .reset_index()
)

# optional: order tags by gen_1 performance
#pivot = pivot.sort_values("gen_3").dropna()

pivot.to_csv("../results/tables/human_mean_pass_rate_by_tag_and_gen_pivot.csv",
           index=False)

pivot

KeyError: 'tag'

## TPR for Gemma - Total ~ pivot

In [ ]:
agg = (
    df[df.model == "gemma"]
      .groupby(["tag", "generation"])
      .agg(
          mean_pass_rate=("pass_rate", "mean"),
          n=("pass_rate", "count")
      )
      .reset_index()
)

# keep only sufficiently large groups
agg = agg[agg.n >= 15].sort_values(["tag", "generation"])

agg.to_csv("../results/tables/gemma_mean_pass_rate_by_tag_and_gen.csv",
           index=False)

pivot = (
    agg.pivot(index="tag", columns="generation", values="mean_pass_rate")
       .reset_index()
)

# optional: order tags by gen_1 performance
pivot = pivot.sort_values("gen_3").dropna()

pivot.to_csv("../results/tables/gemma_mean_pass_rate_by_tag_and_gen_pivot.csv",
           index=False)

pivot

### TPR for Gemma - 1

In [ ]:
agg = (
    df[(df.model=="gemma") & (df.generation=="gen_1")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg

### TPR for Gemma Gen2

In [ ]:
agg = (
    df[(df.model=="gemma") & (df.generation=="gen_2")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg.to_csv("../results/tables/gemma_mean_pass_rate_gen_2.csv", index=False)

agg

### TPR for Gemma Gen3 

In [ ]:
agg = (
    df[(df.model=="gemma") & (df.generation=="gen_3")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg.to_csv("../results/tables/gemma_mean_pass_rate_gen_3.csv", index=False)

agg

## TPR for Llama ~ pivot

In [ ]:
agg = (
    df[df.model == "llama"]
      .groupby(["tag", "generation"])
      .agg(
          mean_pass_rate=("pass_rate", "mean"),
          n=("pass_rate", "count")
      )
      .reset_index()
)

# keep only sufficiently large groups
agg = agg[agg.n >= 15].sort_values(["tag", "generation"])

agg.to_csv("../results/tables/llama_mean_pass_rate_by_tag_and_gen.csv",
           index=False)

pivot = (
    agg.pivot(index="tag", columns="generation", values="mean_pass_rate")
       .reset_index()
)

# optional: order tags by gen_1 performance
pivot = pivot.sort_values("gen_3").dropna()

pivot.to_csv("../results/tables/llama_mean_pass_rate_by_tag_and_gen_pivot.csv",
           index=False)

pivot

### TPR for Llama gen1

In [ ]:
agg = (
    df[(df.model=="llama") & (df.generation=="gen_1")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg.to_csv("../results/tables/llama_mean_pass_rate_gen_1.csv", index=False)

agg

### TPR for llama gen2 

In [ ]:
agg = (
    df[(df.model=="llama") & (df.generation=="gen_2")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg.to_csv("../results/tables/llama_mean_pass_rate_gen_2.csv", index=False)

agg

### TPR for Llama gen3

In [ ]:
agg = (
    df[(df.model=="llama") & (df.generation=="gen_3")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")

agg.to_csv("../results/tables/llama_mean_pass_rate_gen_3.csv", index=False)
agg

## TPR for Qwen total ~ pivot

In [ ]:
agg = (
    df[df.model == "qwen"]
      .groupby(["tag", "generation"])
      .agg(
          mean_pass_rate=("pass_rate", "mean"),
          n=("pass_rate", "count")
      )
      .reset_index()
)

# keep only sufficiently large groups
agg = agg[agg.n >= 15].sort_values(["tag", "generation"])

agg.to_csv("../results/tables/qwen_mean_pass_rate_by_tag_and_gen.csv",
           index=False)

pivot = (
    agg.pivot(index="tag", columns="generation", values="mean_pass_rate")
       .reset_index()
)

# optional: order tags by gen_1 performance
pivot = pivot.sort_values("gen_3").dropna()

pivot.to_csv("../results/tables/qwen_mean_pass_rate_by_tag_and_gen_pivot.csv",
           index=False)

pivot

### TPR for Qwen gen_1

In [ ]:
agg = (
    df[(df.model=="qwen") & (df.generation=="gen_1")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg.to_csv("../results/tables/qwen_mean_pass_rate_gen_1.csv", index=False)

agg

### TPR for Qwen gen2

In [ ]:
agg = (
    df[(df.model=="qwen") & (df.generation=="gen_2")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg.to_csv("../results/tables/qwen_mean_pass_rate_gen_2.csv", index=False)

agg

### TPR for Qwen gen3

In [ ]:
agg = (
    df[(df.model=="qwen") & (df.generation=="gen_3")]
      .groupby("tag")
      .agg(
          mean_pass_rate=("pass_rate","mean"),
          n=("pass_rate","count")
      )
      .reset_index()
)

agg = agg[agg.n >= 10].sort_values("mean_pass_rate")
agg.to_csv("../results/tables/qwen_mean_pass_rate_gen_3.csv", index=False)
agg

# CWE with Tags

## Total

In [ ]:
import pandas as pd

df = pd.read_csv("../results/raw_csv/cppcheck_program_level.csv")

summary = (
    df.groupby(["model","generation"])
      .agg(
          analyzed=("analyzed","sum"),
          vulnerable=("has_cwe","sum")
      )
      .reset_index()
)

summary["fraction"] = summary["vulnerable"] / summary["analyzed"]

summary

## Bootstrap CI for CWE distribution

In [ ]:

df = pd.read_csv("../results/raw_csv/cppcheck_agg.csv")
df

In [ ]:
# scripts/plot_per_cwe_distribution.py
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../results/raw_csv/cppcheck_agg.csv")

# Normalize CWE label (e.g., "CWE-119")
df["cwe_id"] = df["cwe"].astype(str).str.extract(r"(\d+)")

# Drop rows without CWE mapping
df = df.dropna(subset=["cwe_id"])

# Count CWEs per model (global distribution)
cwe_counts = (
    df.groupby(["model", "cwe_id"])
      .size()
      .reset_index(name="count")
)

# Normalize per model
cwe_counts["fraction"] = (
    cwe_counts.groupby("model")["count"]
              .transform(lambda x: x / x.sum())
)


TOP_K = 10

for model, g in cwe_counts.groupby("model"):
    top = g.sort_values("count", ascending=False).head(TOP_K)

    plt.figure(figsize=(6,4))
    plt.barh(top["cwe_id"], top["fraction"])
    plt.xlabel("Fraction of all CWE reports")
    plt.title(f"Top-{TOP_K} CWE distribution — {model}")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"../results/figures/cwe_type_dist_{model}.pdf")
    plt.show()
    plt.close()

In [ ]:
# scripts/mitre_top25_program_prevalence.py
import pandas as pd
import re

MITRE_TOP25 = {
    20,22,59,73,77,78,79,89,94,119,
    120,125,190,200,269,287,295,306,
    352,362,400,416,476,787
}

df = pd.read_csv("../results/raw_csv/cppcheck_agg.csv")

def cwe_num(cwe):
    if pd.isna(cwe): return None
    m = re.search(r"\d+", str(cwe))
    return int(m.group()) if m else None

df["cwe_num"] = df["cwe"].apply(cwe_num)
df["is_top25"] = df["cwe_num"].apply(
    lambda x: x in MITRE_TOP25 if x is not None else False
)

# program-level indicator
prog = (
    df.groupby(["model","generation","instruction_id"])["is_top25"]
      .any()
      .reset_index(name="has_top25")
)

summary = (
    prog.groupby(["model","generation"])
        .agg(
            top25_prevalence=("has_top25","mean"),
            n_programs=("has_top25","count")
        )
        .reset_index()
)

summary.to_csv(
    "../results/tables/mitre_top25_program_prevalence.csv",
    index=False
)

print("Wrote mitre_top25_program_prevalence.csv")

summary

In [ ]:
# scripts/mitre_top25_counts_per_program.py
import pandas as pd
import re

MITRE_TOP25 = {
    20,22,59,73,77,78,79,89,94,119,
    120,125,190,200,269,287,295,306,
    352,362,400,416,476,787
}

df = pd.read_csv("../results/raw_csv/cppcheck_agg.csv")

def cwe_num(cwe):
    if pd.isna(cwe):
        return None
    m = re.search(r"\d+", str(cwe))
    return int(m.group()) if m else None

df["cwe_num"] = df["cwe"].apply(cwe_num)
df["is_top25"] = df["cwe_num"].apply(
    lambda x: x in MITRE_TOP25 if x is not None else False
)

# count Top-25 CWE instances per program
prog_counts = (
    df[df["is_top25"]]
    .groupby(["model","generation","instruction_id"])
    .size()
    .reset_index(name="top25_count")
)

# include programs with zero Top-25 CWEs
programs = (
    df.groupby(["model","generation","instruction_id"])
      .size()
      .reset_index()[["model","generation","instruction_id"]]
)

prog_counts = all_programs.merge(
    prog_counts,
    on=["model","generation","instruction_id"],
    how="left"
).fillna({"top25_count": 0})

prog_counts["top25_count"] = prog_counts["top25_count"].astype(int)

prog_counts.to_csv(
    "../results/tables/mitre_top25_counts_per_program.csv",
    index=False
)

prog_counts = prog_counts[prog_counts['top25_count']>0] 


# group by model and generation: count programs with ≥1 Top-25 CWE
summary = (
    prog_counts
    .groupby(["model", "generation"])
    .agg(
        n_programs_with_top25=("instruction_id", "nunique")
    )
    .reset_index()
)

print(summary)



In [ ]:
vul = pd.read_csv("../results/raw_csv/cppcheck_agg.csv")

vul_flag = (
    vul.groupby(["model","generation","instruction_id"])
       .size()
       .reset_index(name="cwe_count")
)

vul_flag["has_cwe"] = True


df = programs.merge(
    vul_flag[["model","generation","instruction_id","has_cwe"]],
    on=["model","generation","instruction_id"],
    how="left"
)

df["has_cwe"] = df["has_cwe"].fillna(False)



In [ ]:
gen_stats = (
    df.groupby(["model","generation"])
      .agg(
          vulnerable_programs=("has_cwe","sum"),
          total_programs=("has_cwe","count")
      )
      .reset_index()
)

gen_stats["vuln_rate"] = (
    gen_stats["vulnerable_programs"] / gen_stats["total_programs"]
)

gen_stats

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

plt.figure(figsize=(6,4))

ax = sns.barplot(
    data=summary,
    x="generation",
    y="fraction",
    hue="model",
    palette='Set2'
)

plt.title("CWE in Generated Code Analyzed Coding Solutions")

ax.set_ylabel("Fraction of Programs with ≥1 CWE")
ax.set_xlabel("Generation")
ax.set_ylim(0, 1)

plt.legend(title="Model", fontsize='small', title_fontsize='small',)

plt.tight_layout()
plt.savefig("../results/figures/Baseline_CWE.pdf")
plt.show()

### CWE with tags Gemma 1

In [ ]:
inst = (
    vul.groupby("instruction_id")
       .agg(
           has_cwe=("cwe", lambda x: x.notna().any()),
           cwe_list=("cwe", lambda x: set(x.dropna()))
       )
       .reset_index()
)

In [ ]:
tags = pd.read_csv("../results/raw_csv/instruction_tags.csv")

tags["shard"] = tags["shard"].astype(int).map(lambda x: f"{x:05d}")


inst["shard"] = inst["instruction_id"] % 3
inst["shard"] = inst["shard"].map(lambda x: f"{x:05d}")

df = inst.merge(tags, on=["instruction_id","shard"], how="left")
df = df.assign(tag=df["tags"].str.split("|")).explode("tag")

In [ ]:
df_cwe = df.explode("cwe_list").dropna(subset=["cwe_list"])

In [ ]:
tag_cwe_counts = (
    df_cwe.groupby(["tag", "cwe_list"])
          .size()
          .reset_index(name="count")
          .sort_values(["tag", "count"], ascending=[True, False])
)

In [ ]:
tag_cwe_counts = tag_cwe_counts[tag_cwe_counts["count"] >= 5]
tag_cwe_counts

# Summary Statistics & Bootstrap Analysis

## T1: Dataset Summary

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

master = pd.read_csv("../results/raw_csv/master_analysis.csv")

summary = (
    master.groupby(["model", "generation"])
          .agg(
              n_samples=("sample_id", "count"),
              compiled=("compiled", "sum"),
              has_tests=("has_tests", "sum"),
          )
          .reset_index()
)

summary["compile_rate"] = summary["compiled"] / summary["n_samples"]

summary.to_csv("../results/tables/dataset_summary.csv", index=False)
summary

## T2: Pass Rate by Model with Bootstrap CI

In [ ]:
bootstrap_ci = pd.read_csv("../results/raw_csv/bootstrap_ci_results.csv")

pass_rates = bootstrap_ci[bootstrap_ci["metric"].isin(["full_pass_rate", "mean_pass_rate"])]
pass_rates = pass_rates[["metric", "model", "n_samples", "point_estimate", "ci_lower", "ci_upper"]]

pass_rates.to_csv("../results/tables/pass_rate_bootstrap_ci.csv", index=False)
pass_rates

## T3: Human vs AI Pass Rate Differences

In [ ]:
diff_metrics = bootstrap_ci[bootstrap_ci["metric"].str.contains("diff_human")]
diff_table = diff_metrics[["metric", "model", "point_estimate", "ci_lower", "ci_upper", "p_value", "significant_95"]]

diff_table.to_csv("../results/tables/human_vs_ai_differences.csv", index=False)
diff_table

## F1: Pass Rate Bar Chart with Error Bars

In [ ]:
full_pass = bootstrap_ci[bootstrap_ci["metric"] == "full_pass_rate"].copy()
full_pass["error_low"] = full_pass["point_estimate"] - full_pass["ci_lower"]
full_pass["error_high"] = full_pass["ci_upper"] - full_pass["point_estimate"]

model_order = ["human", "gemma", "llama", "qwen"]
full_pass["model"] = pd.Categorical(full_pass["model"], categories=model_order, ordered=True)
full_pass = full_pass.sort_values("model")

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette("Set2", n_colors=4)

bars = ax.bar(
    full_pass["model"],
    full_pass["point_estimate"],
    yerr=[full_pass["error_low"], full_pass["error_high"]],
    capsize=5,
    color=colors,
    edgecolor="black"
)

ax.set_ylabel("Full Pass Rate", fontsize=12)
ax.set_xlabel("Model", fontsize=12)
ax.set_title("Test Pass Rate by Model (100% pass)", fontsize=14)
ax.set_ylim(0, 1)

for bar, val in zip(bars, full_pass["point_estimate"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"{val:.1%}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("../results/figures/pass_rate_by_model.pdf")
plt.savefig("../results/figures/pass_rate_by_model.png", dpi=150)
plt.show()

## T4 & T5: CWE Summary (Compiled & Passing)

In [ ]:
cwe_compiled = bootstrap_ci[bootstrap_ci["metric"] == "mean_cwe_count_compiled"]
cwe_passing = bootstrap_ci[bootstrap_ci["metric"] == "mean_cwe_count_passing"]

cwe_summary = pd.concat([
    cwe_compiled.assign(subset="compiled"),
    cwe_passing.assign(subset="passing")
])

cwe_summary = cwe_summary[["subset", "model", "n_samples", "point_estimate", "ci_lower", "ci_upper"]]
cwe_summary.to_csv("../results/tables/cwe_summary_by_subset.csv", index=False)
cwe_summary

## F2: CWE per File (Passing vs Failing)

In [ ]:
master = pd.read_csv("../results/raw_csv/master_analysis.csv")
detailed = pd.read_csv("../results/raw_csv/cppcheck_detailed.csv")

detailed_cwe = detailed[detailed["cwe"].notna()].copy()
detailed_cwe["sample_id"] = detailed_cwe["sample_id"].apply(
    lambda x: x if x.endswith(".cpp") else x + ".cpp"
)

cwe_counts = detailed_cwe.groupby("sample_id").size().reset_index(name="cwe_count")
master_cwe = master.merge(cwe_counts, on="sample_id", how="left")
master_cwe["cwe_count"] = master_cwe["cwe_count"].fillna(0)

master_cwe["status"] = "not compiled"
master_cwe.loc[master_cwe["compiled"] == True, "status"] = "compiled (failing)"
master_cwe.loc[master_cwe["pass_rate"] == 1.0, "status"] = "passing"

status_order = ["passing", "compiled (failing)", "not compiled"]
master_cwe["status"] = pd.Categorical(master_cwe["status"], categories=status_order, ordered=True)

model_order = ["human", "gemma", "llama", "qwen"]

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=master_cwe,
    x="model",
    y="cwe_count",
    hue="status",
    order=model_order,
    palette="Set2",
    ax=ax,
    errorbar="ci"
)

ax.set_ylabel("Mean CWE Count per File", fontsize=12)
ax.set_xlabel("Model", fontsize=12)
ax.set_title("CWE Findings by Model and Solution Status", fontsize=14)
ax.legend(title="Status")

plt.tight_layout()
plt.savefig("../results/figures/cwe_by_status.pdf")
plt.savefig("../results/figures/cwe_by_status.png", dpi=150)
plt.show()

## T6: Top CWEs by Model (Passing Only)

In [ ]:
passing_ids = set(master[master["pass_rate"] == 1.0]["sample_id"])

detailed_cwe["cwe"] = detailed_cwe["cwe"].astype(int)
df_pass = detailed_cwe[detailed_cwe["sample_id"].isin(passing_ids)]

top_cwes_list = []
for model in ["human", "gemma", "llama", "qwen"]:
    model_df = df_pass[df_pass["model"] == model]
    top5 = model_df["cwe"].value_counts().head(5).reset_index()
    top5.columns = ["cwe", "count"]
    top5["model"] = model
    top5["rank"] = range(1, len(top5) + 1)
    top_cwes_list.append(top5)

top_cwes = pd.concat(top_cwes_list)
top_cwes["cwe_label"] = "CWE-" + top_cwes["cwe"].astype(str)

top_cwes_pivot = top_cwes.pivot(index="rank", columns="model", values="cwe_label")
top_cwes_pivot = top_cwes_pivot[["human", "gemma", "llama", "qwen"]]

top_cwes_pivot.to_csv("../results/tables/top_cwes_by_model_passing.csv")
top_cwes_pivot

## F3: Top CWE Heatmap (Compiled)

In [ ]:
compiled_ids = set(master[master["compiled"] == True]["sample_id"])
df_compiled = detailed_cwe[detailed_cwe["sample_id"].isin(compiled_ids)]

top10_cwes = df_compiled["cwe"].value_counts().head(10).index.tolist()

cwe_model_counts = (
    df_compiled[df_compiled["cwe"].isin(top10_cwes)]
    .groupby(["model", "cwe"])
    .size()
    .reset_index(name="count")
)

n_files_per_model = master[master["compiled"] == True].groupby("model").size()
cwe_model_counts["rate"] = cwe_model_counts.apply(
    lambda row: row["count"] / n_files_per_model[row["model"]], axis=1
)

heatmap_data = cwe_model_counts.pivot(index="cwe", columns="model", values="rate")
heatmap_data = heatmap_data[["human", "gemma", "llama", "qwen"]]
heatmap_data.index = "CWE-" + heatmap_data.index.astype(str)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    ax=ax,
    cbar_kws={"label": "Findings per File"}
)

ax.set_title("Top 10 CWE Rates by Model (Compiled Only)", fontsize=14)
ax.set_xlabel("Model", fontsize=12)
ax.set_ylabel("CWE", fontsize=12)

plt.tight_layout()
plt.savefig("../results/figures/cwe_heatmap.pdf")
plt.savefig("../results/figures/cwe_heatmap.png", dpi=150)
plt.show()

## T7: CWE Stability Across Generations

In [ ]:
stability = pd.read_csv("../results/raw_csv/cwe_stability.csv")

stability_display = stability[[
    "model", "gen1", "gen2", 
    "jaccard_similarity", "spearman_correlation", "bhattacharyya_distance"
]].round(3)

stability_display.to_csv("../results/tables/cwe_stability.csv", index=False)
stability_display

## F5: CWE Stability Bar Chart

In [ ]:
stability_avg = (
    stability.groupby("model")[["jaccard_similarity", "spearman_correlation"]]
             .mean()
             .reset_index()
)

stability_melt = stability_avg.melt(
    id_vars="model",
    value_vars=["jaccard_similarity", "spearman_correlation"],
    var_name="metric",
    value_name="value"
)

model_order = ["human", "gemma", "llama", "qwen"]
stability_melt["model"] = pd.Categorical(stability_melt["model"], categories=model_order, ordered=True)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=stability_melt,
    x="model",
    y="value",
    hue="metric",
    palette="Set2",
    ax=ax
)

ax.set_ylabel("Score", fontsize=12)
ax.set_xlabel("Model", fontsize=12)
ax.set_title("CWE Distribution Stability Across Generations", fontsize=14)
ax.set_ylim(0, 1)
ax.legend(title="Metric", labels=["Jaccard Similarity", "Spearman Correlation"])

plt.tight_layout()
plt.savefig("../results/figures/cwe_stability.pdf")
plt.savefig("../results/figures/cwe_stability.png", dpi=150)
plt.show()

## F6: Bootstrap CI Forest Plot (Human vs AI)

In [ ]:
diff_data = bootstrap_ci[
    bootstrap_ci["metric"].isin([
        "full_pass_diff_human_minus_ai",
        "mean_rate_diff_human_minus_ai",
        "cwe_count_diff_passing"
    ])
].copy()

diff_data["label"] = diff_data["metric"].map({
    "full_pass_diff_human_minus_ai": "Full Pass Rate",
    "mean_rate_diff_human_minus_ai": "Mean Pass Rate",
    "cwe_count_diff_passing": "CWE Count (Passing)"
}) + "\n" + diff_data["model"]

fig, ax = plt.subplots(figsize=(10, 6))

y_pos = range(len(diff_data))
colors = sns.color_palette("Set2", n_colors=3)
color_map = {
    "Full Pass Rate": colors[0],
    "Mean Pass Rate": colors[1],
    "CWE Count (Passing)": colors[2]
}

for i, (_, row) in enumerate(diff_data.iterrows()):
    metric_type = row["label"].split("\n")[0]
    ax.errorbar(
        row["point_estimate"],
        i,
        xerr=[[row["point_estimate"] - row["ci_lower"]], [row["ci_upper"] - row["point_estimate"]]],
        fmt="o",
        color=color_map[metric_type],
        capsize=5,
        markersize=8
    )

ax.axvline(x=0, color="gray", linestyle="--", alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(diff_data["label"])
ax.set_xlabel("Difference (Human - AI)", fontsize=12)
ax.set_title("Bootstrap 95% CIs for Human vs AI Differences", fontsize=14)

plt.tight_layout()
plt.savefig("../results/figures/forest_plot_differences.pdf")
plt.savefig("../results/figures/forest_plot_differences.png", dpi=150)
plt.show()

# Filtered Analysis (Excluding Interactive Problems)

Interactive problems require real-time bidirectional I/O with a judge and cannot be tested offline.
We exclude them to ensure fair comparison across all models.

In [ ]:
# Filter out interactive problems from master dataset
EXCLUDED_TAGS = ["interactive"]

# Reload master and explode tags
master = pd.read_csv("../results/raw_csv/master_analysis.csv")

# Identify samples with excluded tags
def has_excluded_tag(tags_str):
    if pd.isna(tags_str):
        return False
    tags = [t.strip() for t in str(tags_str).split("|")]
    return any(t in EXCLUDED_TAGS for t in tags)

master["is_excluded"] = master["tags"].apply(has_excluded_tag)

# Create filtered dataset
master_filtered = master[~master["is_excluded"]].copy()

print(f"Original samples: {len(master)}")
print(f"Excluded (interactive): {master['is_excluded'].sum()}")
print(f"Filtered samples: {len(master_filtered)}")
print(f"\nExcluded by model:")
print(master[master["is_excluded"]].groupby("model").size())

In [ ]:
# Pass rates excluding interactive problems
filtered_pass = master_filtered.groupby("model").agg(
    samples=('sample_id', 'count'),
    passing=('pass_rate', lambda x: (x == 1.0).sum()),
    mean_pass=('pass_rate', 'mean')
).round(3)

filtered_pass["pass_pct"] = (filtered_pass["passing"] / filtered_pass["samples"] * 100).round(1)

# Compare with original
orig_pass = master.groupby("model").agg(
    orig_samples=('sample_id', 'count'),
    orig_passing=('pass_rate', lambda x: (x == 1.0).sum())
)
orig_pass["orig_pass_pct"] = (orig_pass["orig_passing"] / orig_pass["orig_samples"] * 100).round(1)

comparison = filtered_pass.join(orig_pass[["orig_pass_pct"]])
comparison["improvement"] = comparison["pass_pct"] - comparison["orig_pass_pct"]

comparison.to_csv("../results/tables/pass_rate_filtered_vs_original.csv")
print("100% Pass Rate: Original vs Filtered (excl. interactive)")
comparison[["orig_pass_pct", "pass_pct", "improvement"]]

In [ ]:
# Bar chart: Filtered pass rates
model_order = ["human", "gemma", "llama", "qwen"]
plot_data = comparison.loc[model_order].reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette("Set2", n_colors=4)

bars = ax.bar(plot_data["model"], plot_data["pass_pct"], color=colors, edgecolor="black")

ax.set_ylabel("100% Pass Rate (%)", fontsize=12)
ax.set_xlabel("Model", fontsize=12)
ax.set_title("Test Pass Rate by Model (Excluding Interactive Problems)", fontsize=14)
ax.set_ylim(0, 100)

for bar, val in zip(bars, plot_data["pass_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{val:.1f}%", ha="center", fontsize=11)

plt.tight_layout()
plt.savefig("../results/figures/pass_rate_filtered.pdf")
plt.savefig("../results/figures/pass_rate_filtered.png", dpi=150)
plt.show()

In [ ]:
# Tag-wise pass rates (filtered)
master_exploded_filt = master_filtered.assign(
    tag=master_filtered["tags"].str.split("|")
).explode("tag")
master_exploded_filt["tag"] = master_exploded_filt["tag"].str.strip()

# Pass rate by tag and model
tag_pass_filt = master_exploded_filt.groupby(["tag", "model"]).agg(
    total=('sample_id', 'count'),
    passing=('pass_rate', lambda x: (x == 1.0).sum())
).reset_index()
tag_pass_filt["pass_pct"] = (tag_pass_filt["passing"] / tag_pass_filt["total"] * 100).round(1)

# Common tags
common_tags = master_exploded_filt["tag"].value_counts().head(20).index.tolist()
tag_pass_common = tag_pass_filt[tag_pass_filt["tag"].isin(common_tags)]

# Pivot
tag_pivot_filt = tag_pass_common.pivot(index="tag", columns="model", values="pass_pct")
tag_pivot_filt = tag_pivot_filt[["human", "gemma", "llama", "qwen"]]

tag_pivot_filt.to_csv("../results/tables/pass_rate_by_tag_filtered.csv")
print("Pass Rate by Tag (Excluding Interactive):")
tag_pivot_filt.sort_values("human", ascending=False)

In [ ]:
# Human-AI gap analysis (filtered)
tag_pivot_filt["ai_avg"] = tag_pivot_filt[["gemma", "llama", "qwen"]].mean(axis=1)
tag_pivot_filt["human_ai_gap"] = tag_pivot_filt["human"] - tag_pivot_filt["ai_avg"]

gap_sorted = tag_pivot_filt.sort_values("human_ai_gap", ascending=False)

gap_sorted[["human", "ai_avg", "human_ai_gap"]].to_csv(
    "../results/tables/human_ai_gap_filtered.csv"
)

print("Tags with Largest Human-AI Gap (Filtered):")
gap_sorted[["human", "ai_avg", "human_ai_gap"]].head(15)

In [ ]:
# Summary: Overall filtered statistics
print("="*60)
print("FILTERED ANALYSIS SUMMARY (Excluding Interactive Problems)")
print("="*60)
print(f"\nTotal samples analyzed: {len(master_filtered)}")
print(f"Interactive problems excluded: {master['is_excluded'].sum()}")
print(f"\n100% Pass Rates:")
for model in ["human", "gemma", "llama", "qwen"]:
    pct = comparison.loc[model, "pass_pct"]
    orig = comparison.loc[model, "orig_pass_pct"]
    print(f"  {model}: {pct:.1f}% (was {orig:.1f}%)")
print(f"\nHuman-AI gap (filtered): {comparison.loc['human', 'pass_pct'] - comparison.loc[['gemma', 'llama', 'qwen'], 'pass_pct'].mean():.1f} pp")

In [ ]:
# Export filtered results to LaTeX
latex_df = comparison[["samples", "passing", "pass_pct"]].copy()
latex_df.columns = ["Samples", "Passing", "Pass Rate (\%)"]

to_latex_booktabs(
    latex_df,
    "pass_rate_filtered.tex",
    "Test Pass Rates by Model (Excluding Interactive Problems)",
    "tab:pass_rate_filtered"
)